### Save MRMS data for different variables and date ranges

### Begin home location to MRMS matching

#### Load home location file

### Load MRMS variable files

### Match home locations to nearest MRMS data

In [19]:
from plotly.subplots import make_subplots
import plotly
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
plotly.offline.init_notebook_mode(connected=True)


In [2]:
mrms_fn = "/glade/campaign/cisl/aiml/cat_flood/mrms_data/2024/veraset_statistics_mrms_20250721.parquet"
mrms_df = pd.read_parquet(mrms_fn)

In [3]:
data_dir = "/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20250720/"
fns = [f"veraset_2024-03-{date:02}_PST_chunked.parquet" for date in range(29,32)]
fns += [f"veraset_2024-04-{date:02}_PST_chunked.parquet" for date in range(1,2)]
metrics_df = pd.concat(
    pd.read_parquet(data_dir+pf)
    for pf in fns)

In [4]:
matching_uids = np.intersect1d(metrics_df.uid, mrms_df.index)
matching_uids.shape

(17024,)

In [5]:
all_data = metrics_df.merge(mrms_df, right_index=True, left_on='uid')
all_data.shape

(64316, 29)

In [7]:
mrms_df.precip_max.describe()

count    19197.000000
mean         5.400781
std          2.641675
min          0.000000
25%          3.600000
50%          5.000000
75%          6.700000
max         19.400000
Name: precip_max, dtype: float64

In [22]:
mrms_df["precip_max_category"] =  pd.cut(mrms_df['precip_max'], bins=[0, 5, float('Inf')], labels=['low', 'high'])

In [28]:
def get_precip_max_category(uid, high_uids):
    if uid in high_uids:
        return 'high'
    else:
        return 'low'

In [29]:
high_uids = mrms_df[mrms_df.precip_max_category == 'high'].index
metrics_df["precip_max_category"] = metrics_df.uid.apply(get_precip_max_category, high_uids=high_uids)

In [99]:
metrics = ['number_of_visits', 'distance_straight_line',
           'radius_of_gyration', 'num_stops', 'max_distance_from_home']
labels = ['Number of Visits ', 'Cumulative Distance',
           'Radius of Gyration', 'Number of Stops', 'Maxmimum Distance from Home']

for i, metric in enumerate(metrics):
    fig = make_subplots(rows=2, cols=2,
                        specs=[[{"colspan": 2}, None],
                               [{}, {}]],
        subplot_titles=("Max Precipitation (MRMS)", "", ""),
        x_title=f"{labels[i]}",
        shared_yaxes=True,
        horizontal_spacing=0.01,
        vertical_spacing=0.1)
    
    fig.add_trace(
        go.Box(x=mrms_df[mrms_df["precip_max_category"] == "high"]["precip_max_category"],
               y=mrms_df[mrms_df["precip_max_category"] == "high"]["precip_max"],
               marker_color='#FF4136', showlegend=False),
        row=1, col=1
    )
    fig.add_trace(
        go.Box(x=mrms_df[mrms_df["precip_max_category"] == "low"]["precip_max_category"],
               y=mrms_df[mrms_df["precip_max_category"] == "low"]["precip_max"],
               marker_color='#3D9970', showlegend=False),
        row=1, col=1
    )
    fig.add_trace(
        go.Box(x=metrics_df[metrics_df["precip_max_category"] == "high"]["date"],
               y=metrics_df[metrics_df["precip_max_category"] == "high"][metric],
               marker_color='#FF4136', showlegend=False),
        row=2, col=1
    )
    fig.add_trace(
        go.Box(x=metrics_df[metrics_df["precip_max_category"] == "low"]["date"],
               y=metrics_df[metrics_df["precip_max_category"] == "low"][metric],
               marker_color='#3D9970', showlegend=False),
        row=2, col=2
    )

    fig.update_layout(autosize=False, width=800, height=600,
                      font_size=16, margin=dict(l=20,r=20,b=60,t=40))
    fig.write_html(f"html/{metrics[i]}.html")
    fig.show()